In [3]:
import pandas as pd
import numpy as np
import nltk
import re
import torch
import torch.nn as nn
import torch.optim as optim
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset, DataLoader, TensorDataset

nltk.download('punkt')
nltk.download('stopwords')

df = pd.read_csv("../data/train.csv")
df = df.dropna(subset=['text', 'class'])

stop_words = set(stopwords.words("catalan"))

def preprocess(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r"[^\w\s]", '', text)
    tokens = word_tokenize(text, language='spanish') #no es català però kinda close 
    tokens = [word for word in tokens if word not in stop_words and len(word) > 2]
    return tokens

df['tokens'] = df['text'].apply(preprocess)
w2v_model = Word2Vec(sentences=df['tokens'], vector_size=250, window=5, min_count=3, workers=4, epochs=20)

def get_doc_vector(tokens):
    vecs = [w2v_model.wv[t] for t in tokens if t in w2v_model.wv]
    if vecs:
        return np.mean(vecs, axis=0)
    else:
        return np.zeros(w2v_model.vector_size)

df['doc_vec'] = df['tokens'].apply(get_doc_vector)

X = np.vstack(df['doc_vec'].values)
le = LabelEncoder()
y = le.fit_transform(df['class'])
num_classes = len(np.unique(y))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

class TextClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(TextClassifier, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.dropout(out)
        return self.fc2(out)

model = TextClassifier(input_dim=250, hidden_dim=256, output_dim=num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(30):
    model.train()
    total_loss = 0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: Loss = {total_loss:.4f}")

model.eval()
correct, total = 0, 0
with torch.no_grad():
    for xb, yb in test_loader:
        preds = model(xb)
        _, predicted = torch.max(preds, 1)
        correct += (predicted == yb).sum().item()
        total += yb.size(0)

print(f"\nTest Accuracy: {correct / total:.4f}")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\arhip\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\arhip\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Epoch 1: Loss = 1256.7979
Epoch 2: Loss = 940.2710
Epoch 3: Loss = 869.0727
Epoch 4: Loss = 824.9649
Epoch 5: Loss = 788.4018
Epoch 6: Loss = 762.1691
Epoch 7: Loss = 737.1084
Epoch 8: Loss = 716.9089
Epoch 9: Loss = 700.0278
Epoch 10: Loss = 682.3251
Epoch 11: Loss = 673.7686
Epoch 12: Loss = 657.3067
Epoch 13: Loss = 643.1497
Epoch 14: Loss = 629.2520
Epoch 15: Loss = 621.1835
Epoch 16: Loss = 607.3149
Epoch 17: Loss = 593.9529
Epoch 18: Loss = 600.0435
Epoch 19: Loss = 583.1262
Epoch 20: Loss = 575.7297
Epoch 21: Loss = 563.5472
Epoch 22: Loss = 560.6240
Epoch 23: Loss = 555.7215
Epoch 24: Loss = 551.1954
Epoch 25: Loss = 545.7804
Epoch 26: Loss = 534.0818
Epoch 27: Loss = 530.3367
Epoch 28: Loss = 522.9631
Epoch 29: Loss = 525.8468
Epoch 30: Loss = 516.5356

Test Accuracy: 0.9537
